# M7 · 검색이 빗나간 질문 확인
M6 점수를 낸 뒤 실패 질문부터 원인을 본다. 점수만 올리려고 질문이나 정답 문서 ID를 바꾸지는 않는다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import chromadb
from sklearn.feature_extraction.text import HashingVectorizer
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
P=ROOT/'data'/'personal'; E=P/'evaluation'; E.mkdir(parents=True,exist_ok=True)
meta=json.loads((P/'corpus'/'index_meta.json').read_text(encoding='utf-8'))
collection=chromadb.PersistentClient(path=str(ROOT/'chroma_db'/'personal')).get_collection(meta['collection'],embedding_function=None)
vectorizer=HashingVectorizer(analyzer='char',ngram_range=(2,4),n_features=1024,alternate_sign=False,norm='l2',dtype=np.float32)
def search(query,k=5,source_type='kosha_guide'):
    vector=vectorizer.transform([query]).toarray().astype(np.float32)
    found=collection.query(query_embeddings=vector.tolist(),n_results=min(100,collection.count()),
                           where={'source_type':source_type},include=['metadatas','distances'])
    hits=[];seen=set()
    for doc,distance in zip(found['metadatas'][0],found['distances'][0]):
        if doc['doc_id'] in seen:continue
        hits.append({'doc_id':doc['doc_id'],'score':round(1-float(distance),4)})
        seen.add(doc['doc_id'])
        if len(hits)==k:break
    return hits
def measure(rows):
    details=[]
    for row in rows:
        hits=search(row['query'],source_type=row['source_type'])
        ranked=[h['doc_id'] for h in hits]
        gold=set(row['gold_doc_ids'].split('|'))
        found=[i+1 for i,doc_id in enumerate(ranked) if doc_id in gold]
        details.append({'qid':row['qid'],'query':row['query'],'gold':row['gold_doc_ids'],
                        'top5':'|'.join(ranked),'hit5':int(bool(found)),
                        'precision5':len(found)/5,'rr':1/min(found) if found else 0})
    result=pd.DataFrame(details)
    scores={name:round(float(result[name].mean()),4) for name in ('hit5','precision5','rr')}
    return result,scores

## 실패 질문과 원본 청크 상태

In [ ]:
questions=pd.read_csv(E/'questions.csv',encoding='utf-8-sig')
before=pd.read_csv(E/'baseline_details.csv',encoding='utf-8-sig')
failed=before[before.hit5==0].copy()
chunk_rows=[json.loads(line) for line in (P/'corpus'/'chunks.jsonl').read_text(encoding='utf-8').splitlines() if line]
counts=pd.Series([x['doc_id'] for x in chunk_rows]).value_counts()
failed['gold_chunk_count']=failed.gold.map(counts).fillna(0).astype(int)
display(failed[['qid','query','gold','top5','gold_chunk_count']])
failed.to_csv(E/'failure_cases.csv',index=False,encoding='utf-8-sig')

## 실패 사례 점검

- 10개 안전지침 질문 중 기준선에서 4개는 정답 문서가 상위 5개에 없었다. PDF가 코퍼스에 있는지와 실제 검색 가능한 청크가 있는지를 나눠 확인했다. 파일 존재만으로 검색 가능하다고 보지 않았다.
- 원인 후보는 질문과 지침의 표현 차이, 검색 결과에 유사하지만 정답이 아닌 문서가 먼저 올라오는 문제, 청크 경계와 OCR 품질 등으로 정리했다. 원인별로 정답 문서·검색 순위·청크를 같이 살펴 개선 대상을 골랐다.
- 10개 문항 분석으로 모든 검색 실패를 설명할 수는 없다. 이 분류는 사례를 읽고 정한 원인 후보이며 인과적 증명은 아니다.

### 실험 전 고정한 것

- 질문·정답 문서·상위 K·검색 인덱스를 유지하고 한 번에 바꾸는 검색 설정을 제한한다.
- 검색 실패 사례를 평가 데이터에 맞춰 고쳐버리지 않는다. ⑦ 용어사전의 제한된 유의어 확장만 M7 실험 대상으로 둔다.
